In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

debug = True

if debug:
    silver_catalog = "silver"
    silver_schema = "sistema_leitos"
    gold_catalog = "gold"
    gold_schema = "sistema_leitos"
else:
    # Em um ambiente de produção, estes valores viriam de widgets do Databricks
    silver_catalog = dbutils.widgets.get("silver_catalog")
    silver_schema = dbutils.widgets.get("silver_schema")
    gold_catalog = dbutils.widgets.get("gold_catalog")
    gold_schema = dbutils.widgets.get("gold_schema")

In [ ]:
fact_leitos = spark.read.table(f"{silver_catalog}.{silver_schema}.fact_leitos")
dim_estabelecimento = spark.read.table(f"{silver_catalog}.{silver_schema}.dim_estabelecimento")
dim_tempo = spark.read.table(f"{silver_catalog}.{silver_schema}.dim_tempo")

latest_partition = spark.read.table(f"{silver_catalog}.{silver_schema}.fact_leitos").select(F.max("data_extracao")).first()[0]

df_base = (
    fact_leitos
    .filter(F.col("data_extracao") == latest_partition)
    .join(dim_estabelecimento, "cnes")
    .join(dim_tempo, "id_tempo")
).cache()

print(f"Tabelas da camada Prata carregadas para a data de extração: {latest_partition}. Unificação concluída.")

In [ ]:
print("Iniciando a criação de 'agg_capacidade_geografica_mensal'...")
df_geo = (
    df_base
    .groupBy("regiao", "uf", "ano", "mes")
    .agg(
        F.countDistinct("cnes").alias("total_estabelecimentos"),
        F.sum("qtd_leitos_existentes").alias("soma_leitos_existentes"),
        F.sum("qtd_leitos_sus").alias("soma_leitos_sus"),
        F.sum("qtd_uti_total_exist").alias("soma_uti_total_exist"),
        F.sum("qtd_uti_total_sus").alias("soma_uti_total_sus")
    )
    .withColumn(
        "percentual_cobertura_sus",
        F.when(F.col("soma_leitos_existentes") > 0, (F.col("soma_leitos_sus") / F.col("soma_leitos_existentes")) * 100).otherwise(0)
    )
)

target_table = f"{gold_catalog}.{gold_schema}.agg_capacidade_geografica_mensal"

(
    df_geo.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano", "mes")
    .saveAsTable(target_table)
)
print(f"Tabela '{target_table}' atualizada com sucesso.")

In [ ]:
print(f"Otimizando a tabela {target_table}...")

spark.sql(f"OPTIMIZE {target_table} ZORDER BY (regiao, uf)")
print("Otimização concluída.")

In [ ]:
print("Iniciando a criação de 'agg_capacidade_por_tipo_gestao_mensal'...")
df_tipo_gestao = (
    df_base
    .groupBy("desc_natureza_juridica", "tipo_gestao", "ano", "mes")
    .agg(
        F.countDistinct("cnes").alias("total_estabelecimentos"),
        F.sum("qtd_leitos_sus").alias("soma_leitos_sus"),
        F.sum("qtd_uti_pediatrico_sus").alias("soma_uti_pediatrico_sus"),
        F.sum("qtd_uti_neonatal_sus").alias("soma_uti_neonatal_sus")
    )
)

target_table = f"{gold_catalog}.{gold_schema}.agg_capacidade_por_tipo_gestao_mensal"
(
    df_tipo_gestao.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano", "mes")
    .saveAsTable(target_table)
)
print(f"Tabela '{target_table}' atualizada com sucesso.")

In [ ]:
print(f"Otimizando a tabela {target_table}...")
spark.sql(f"OPTIMIZE {target_table} ZORDER BY (desc_natureza_juridica, tipo_gestao)")
print("Otimização concluída.")

In [ ]:
print("Iniciando a criação de 'agg_evolucao_temporal_leitos_nacional'...")

full_fact_leitos = spark.read.table(f"{silver_catalog}.{silver_schema}.fact_leitos")
full_base = full_fact_leitos.join(dim_tempo, "id_tempo")

df_monthly_agg = (
    full_base
    .groupBy("ano", "mes", "id_tempo")
    .agg(
        F.sum("qtd_leitos_sus").alias("total_leitos_sus_nacional"),
        F.sum("qtd_uti_adulto_sus").alias("total_uti_adulto_sus_nacional"),
        F.sum("qtd_uti_pediatrico_sus").alias("total_uti_pediatrico_sus_nacional")
    )
    .orderBy("id_tempo")
)

window_mom = Window.orderBy("id_tempo")

df_temporal = df_monthly_agg.withColumn(
    "leitos_mes_anterior", F.lag("total_leitos_sus_nacional", 1).over(window_mom)
).withColumn(
    "leitos_ano_anterior", F.lag("total_leitos_sus_nacional", 12).over(window_mom)
)

df_temporal_final = df_temporal.withColumn(
    "variacao_mensal_leitos_sus",
    F.when(F.col("leitos_mes_anterior") > 0, (F.col("total_leitos_sus_nacional") - F.col("leitos_mes_anterior")) / F.col("leitos_mes_anterior") * 100).otherwise(0)
).withColumn(
    "variacao_anual_leitos_sus",
    F.when(F.col("leitos_ano_anterior") > 0, (F.col("total_leitos_sus_nacional") - F.col("leitos_ano_anterior")) / F.col("leitos_ano_anterior") * 100).otherwise(0)
).drop("leitos_mes_anterior", "leitos_ano_anterior")

target_table = f"{gold_catalog}.{gold_schema}.agg_evolucao_temporal_leitos_nacional"
(
    df_temporal_final.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano", "mes")
    .saveAsTable(target_table)
)
print(f"Tabela '{target_table}' recriada com sucesso.")

In [ ]:
print(f"Otimizando a tabela {target_table}...")
spark.sql(f"OPTIMIZE {target_table} ZORDER BY (id_tempo)")
print("Otimização concluída.")

In [ ]:
print("Iniciando a criação de 'agg_ranking_estabelecimentos_municipios'...")

latest_month_id = df_base.select(F.max("id_tempo")).first()[0]

df_latest = df_base.filter(F.col("id_tempo") == latest_month_id)

df_ranking = (
    df_latest
    .groupBy("cnes", "nome_estabelecimento", "municipio", "uf")
    .agg(
        F.sum("qtd_leitos_sus").alias("total_leitos_sus"),
        F.sum("qtd_uti_total_sus").alias("total_uti_sus"),
        F.sum("qtd_leitos_existentes").alias("total_leitos_existentes")
    )
    .withColumn("percentual_dependencia_sus", F.when(F.col("total_leitos_existentes") > 0, (F.col("total_leitos_sus") / F.col("total_leitos_existentes")) * 100).otherwise(0))
)

window_ranking = Window.orderBy(F.col("total_uti_sus").desc())

df_ranking_final = df_ranking.withColumn("ranking_nacional_uti", F.dense_rank().over(window_ranking))

target_table = f"{gold_catalog}.{gold_schema}.agg_ranking_estabelecimentos_municipios"


df_ranking_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
print(f"Tabela '{target_table}' criada com sucesso.")

In [ ]:
print(f"Otimizando a tabela {target_table}...")
spark.sql(f"OPTIMIZE {target_table} ZORDER BY (uf, municipio)")
print("Otimização concluída.")

In [ ]:
df_base.unpersist()
print("Camada Ouro (Gold) otimizada e implementada com sucesso.")